<a href="https://colab.research.google.com/github/mteuslms/AI-ML-projects/blob/main/PracticeTinyGPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as Func
import math

In [ ]:
#switch to gpu
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [ ]:
#class EmbeddingTokens
class EmbeddingTokens(nn.Module):
  def __init__(self, vocab_size, d_model):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, d_model)

  def forward(self, x):
    return self.embedding(x)

In [ ]:
#Class Position Encoding
class PositionalEncoding(nn.Module):
  def __init__(self, d_model, max_len=5000):
    super().__init__()
    pe = torch.zeros(max_len, d_model)
    position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    pe = pe.unsqueeze(0)
    self.register_buffer('pe', pe)

  def forward(self, x):
    return x + self.pe[:, :x.size(1)]

In [ ]:
#class Self-Attention
class SelfAttention(nn.Module):
  def __init__(self, d_model):
    super().__init__()
    self.q = nn.Linear(d_model, d_model)
    self.k = nn.Linear(d_model, d_model)
    self.v = nn.Linear(d_model, d_model)

  def forward(self,x):
    Q = self.q(x)
    K = self.k(x)
    V = self.v(x)

    scores = Q @ K.transpose(-2,-1) / (x.size(-1)**(0.5))

    #casual mask
    seq_len = x.size(1) # number of tokens in the input
    mask = torch.tril(torch.ones(seq_len, seq_len, device=x.device))
    scores = scores.masked_fill(mask == 0, float('-inf'))

    attention = Func.softmax(scores, dim=-1)
    return attention @ V

In [ ]:
# MLP Class
class MLP(nn.Module):
  def __init__(self, d_model):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(d_model, 4 * d_model),
        nn.ReLU(),
        nn.Linear(4 * d_model, d_model)
    )

  def forward(self, x):
    return self.net(x)

In [ ]:
#class Block
class Block(nn.Module):
  def __init__(self, d_model):
    super().__init__()
    self.attention = SelfAttention(d_model)
    self.mlp = MLP(d_model)
    self.ln1 = nn.LayerNorm(d_model)
    self.ln2 = nn.LayerNorm(d_model)

  def forward(self, x):
    x = x + self.ln1(self.attention(x))
    x = x + self.ln2(self.mlp(x))
    return x

In [ ]:
class TinyGPT(nn.Module):
  def __init__(self, vocab_size, d_model, n_blocks):
    super().__init__()
    self.embedding = EmbeddingTokens(vocab_size, d_model)
    self.positional_encoding = PositionalEncoding(d_model)
    self.blocks = nn.ModuleList([Block(d_model) for _ in range(n_blocks)])
    self.out = nn.Linear(d_model, vocab_size)

  def forward(self, x):
    x = self.embedding(x)
    x = self.positional_encoding(x)
    for block in self.blocks:
      x = block(x)
    x = self.out(x)
    return x

In [ ]:
# prepare a tiny vocabury
# tiny vocabulary
word_to_id = {
    "<pad>": 0,
    "<eos>": 1,
    "I": 2,
    "love": 3,
    "this": 4,
    "AI": 5,
    'class': 6,
    '床': 7,
    '前': 8,
    '明': 9,
    '月': 10,
    '光': 11,
    'Dr.': 12,
    'Hong': 13,
    'says': 14,
}

id_to_word = {v: k for k, v in word_to_id.items()}
vocab_size = len(word_to_id)

# must train all the shifted sentences in GPT
sentences = [
    ["I", "love", "this", "<eos>"],
    ["love", "this", "AI", "<eos>"],
    ['this', 'AI', 'class'],
    ['床', '前', '明', "<eos>"],
    ['前', '明','月', "<eos>"],
    ['明', '月', '光', "<eos>"],
    ['Dr.', 'Hong', 'says', 'I', 'love', 'this', 'AI', 'class', "<eos>"],
]

data = []
for s in sentences:
    data.append(torch.tensor([word_to_id[w] for w in s]))

# stack into batch
data = torch.nn.utils.rnn.pad_sequence(data, batch_first=True)


In [ ]:
# Create the model
model = TinyGPT(vocab_size=vocab_size, d_model=32, n_blocks=2)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [ ]:
# GPT training
for epoch in range(500):
    # This statement shows that weights can be ignored
    logits = model(data)

    # shift
    logits = logits[:, :-1, :]
    targets = data[:, 1:]

    loss = Func.cross_entropy(
        logits.reshape(-1, vocab_size),
        targets.reshape(-1)
    )

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 100 == 0:
        print(f"epoch {epoch}, loss = {loss.item():.4f}")


epoch 0, loss = 4.6856
epoch 100, loss = 0.0002
epoch 200, loss = 0.0001
epoch 300, loss = 0.0001
epoch 400, loss = 0.0000


In [ ]:
# Text generation function.
def generate(model, start_words, max_new_tokens=5):
    model.eval()

    tokens = [word_to_id[w] for w in start_words]

    for _ in range(max_new_tokens):
        x = torch.tensor([tokens])

        logits = model(x)
        next_token_logits = logits[:, -1, :]

        # decoding, only pick the token with the highest probability
        next_token = torch.argmax(next_token_logits, dim=-1).item()

        tokens.append(next_token)

        if next_token == word_to_id["<eos>"]:
            break

    return [id_to_word[t] for t in tokens]

In [ ]:
# Try some examples
print(generate(model, ["I"]))
print(generate(model, ["love"]))
print(generate(model, ["love", 'this']))
print(generate(model, ['this', 'AI']))
print(generate(model, ["床"]))
print(generate(model, ['前', '明']))
print(generate(model, ['明']))

# can you notice that the GPT model can predict more than one ward forward?
# At this tiny level, this is one major difference between GPT and Transformer

['I', 'love', 'this', '<eos>']
['love', 'this', 'AI', '<eos>']
['love', 'this', 'AI', '<eos>']
['this', 'AI', 'class', '<pad>', '<pad>', '<pad>', '<pad>']
['床', '前', '明', '<eos>']
['前', '明', '月', '<eos>']
['明', '月', '光', '<eos>']


In [ ]:
print(' '.join(generate(model, ["Dr.", 'Hong'])))
# since max_new_tokens=5
# you will see the followings:
# Dr. Hong says I love this AI


Dr. Hong says I love this AI
